# 摘要缓冲记忆（Summary Buffer Memory）

> **一种混合记忆：近期消息逐字保留，更早的历史压缩成摘要。**

想想你是怎么记住和朋友的一通长途电话的。最后几句话？你几乎能逐字复述。那是你的短期记忆。更早的部分？你记得大概意思，记不清原话。那是你的长期记忆。

摘要缓冲记忆的工作方式相同。它把最近的消息按原样保存在一个**缓冲区**里。更早的内容全部压缩成一份**滚动摘要（running summary）**。每次调用时，LLM 同时拿到两样东西：摘要提供历史背景，原始的近期消息保证精确。

纯缓冲记忆（存储每条消息）回忆精确，但每一轮都更费 token。纯摘要记忆（全部压缩）省 token，但丢失近期的细节。摘要缓冲记忆结合了两者的长处。它对客服机器人、教练型智能体和项目管理助手特别有价值。这类智能体既要记住对话的完整脉络，又要精确回应最新一条消息。

核心的工程难题是**转换阈值（transition threshold）**：消息从缓冲区移出、并入摘要的那个临界点。设得太高会浪费 token。设得太低会丢掉重要细节。

**读完本 notebook 你会理解：**
- 如何用 OpenAI SDK 从零构建摘要缓冲记忆。
- 增量摘要（incremental summarization）如何工作：用新淘汰的消息更新摘要。
- 如何调整缓冲区大小，并观察它对 token 用量的影响。
- 这项技术什么时候赢，什么时候悄悄失效。

## 关键概念

- **缓冲区（buffer zone）**：逐字存储的最近 *k* 条消息。它们为即时推理保留精确措辞、语气和细节。
- **滚动摘要**：一段短文字，概括所有已移出缓冲区的消息中的关键事实和主题。这份摘要由 LLM 生成。
- **上下文窗口（context window）**：模型一次调用能读取的最大 token 数（token 是模型内部使用的词片段）。例如，GPT-4o 支持最多 128k token。
- **转换阈值**：消息从缓冲区移入摘要的触发点。可以按消息条数计，也可以按 token 数计。
- **增量摘要**：每次不重新总结全部历史，而是拿现有摘要和新淘汰的消息，用一次 LLM 调用生成更新后的摘要。这样摘要成本保持在低位。
- **token 预算分配（token budget allocation）**：在摘要和缓冲区之间切分上下文窗口的可用 token。典型分配是摘要留 20-30%，近期消息留 70-80%。
- **提示词组装（prompt assembly）**：把 `[system prompt] + [summary] + [recent buffer messages]` 拼成一条提示词发给 LLM 的过程。

## 架构

<p align="center">
  <img src="../../images/diagrams/04_summary_buffer_memory.svg" alt="摘要缓冲记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    User["User Message"] --> Buffer["Buffer\n(recent K messages)"]
    Buffer --> ThresholdCheck{"Buffer exceeds\nthreshold?"}
    ThresholdCheck -- No --> PromptAssembly["Prompt Assembly"]
    ThresholdCheck -- Yes --> Evict["Evict oldest\nmessages"]
    Evict --> Summarizer["Summarizer\n(LLM call)"]
    Summarizer --> SummaryStore["Summary Store\n(running summary)"]
    SummaryStore --> PromptAssembly
    Buffer --> PromptAssembly
    PromptAssembly --> LLM["LLM\n[summary] + [buffer]"]
    LLM --> Response["Response"]
    Response --> Buffer
```

</details>

数据流程是这样的。新的用户消息进入缓冲区。当缓冲区超过配置的阈值，就淘汰最旧的消息。这些消息送进一次摘要 LLM 调用，更新滚动摘要。最终的提示词把系统提示词、摘要和近期缓冲消息拼接在一起。LLM 作出回应，回应又回到缓冲区。每一轮都重复这个循环。

## 环境准备

安装依赖并配置 API 访问。你需要在 `.env` 文件里设置 `LLM_API_KEY` 环境变量。

In [ ]:
%pip -q install python-dotenv tiktoken deepagents langchain-openai

导入 OpenAI SDK、`tiktoken`（OpenAI 的分词器库，用于统计 token 数）和标准库工具。

In [ ]:
import os
import json
import copy
import tiktoken

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 核心实现

我们将构建一个 `SummaryBufferMemory` 类，它：
1. 把近期消息存进缓冲区（一个普通的 Python 列表）。
2. 维护一份较旧历史的滚动摘要。
3. 当缓冲区超过 token 阈值时，淘汰最旧的消息。
4. 用一次 LLM 调用把淘汰的消息并入摘要。
5. 按 `[system] + [summary] + [buffer]` 组装最终提示词。

### token 计数辅助函数

我们需要一种方法统计一组消息占多少 token。这能告诉我们缓冲区什么时候长得太大。

In [ ]:
def count_message_tokens(messages: list[dict], enc: tiktoken.Encoding = encoder) -> int:
    """统计一组聊天消息的总 token 数。

    每条消息都有角色格式化带来的额外 token。
    我们给每条消息加 4 个 token 作为这部分开销的近似值。
    """
    total = 0
    for msg in messages:
        total += 4  # role 和格式化开销
        total += len(enc.encode(msg["content"]))
    return total

### 摘要生成器

这个函数拿当前的滚动摘要和一批新淘汰的消息，让 LLM 生成一份融合了新信息的更新摘要。

可以把它想成更新会议记录。你手上有昨天的记录（现有摘要）和今天的讨论要点（淘汰出来的消息），把它们合并成一份简洁的文档。

In [ ]:
SUMMARY_SYSTEM_PROMPT = """You are a conversation summarizer. Your job is to maintain
a running summary of a conversation. You will receive:
1. The current summary (may be empty if this is the start).
2. New messages that need to be incorporated.

Produce an updated summary that:
- Preserves all key facts, names, preferences, and decisions.
- Stays concise (aim for 3-6 sentences).
- Uses third person ("The user said...", "The assistant suggested...").
- Drops filler and small talk while keeping substance.
"""


def update_summary(
    current_summary: str,
    evicted_messages: list[dict],
) -> str:
    """用一次 LLM 调用把淘汰的消息并入滚动摘要。"""
    # 把淘汰的消息格式化成可读文本
    evicted_text = "\n".join(
        f"{msg['role'].upper()}: {msg['content']}" for msg in evicted_messages
    )

    user_prompt = (
        f"Current summary:\n{current_summary if current_summary else '(empty)'}\n\n"
        f"New messages to incorporate:\n{evicted_text}\n\n"
        f"Write the updated summary:"
    )

    response = llm.invoke([
        {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])
    return to_text(response.content)


### SummaryBufferMemory 类

这是这项技术的核心。这个类管理两个记忆区域：
- **缓冲区**：近期消息的列表（逐字保存）。
- **摘要**：一个字符串，概括更早的历史。

调用 `chat()` 时，类会检查缓冲区是否超过 `max_buffer_tokens`。如果超过，就淘汰最旧的消息并入摘要。然后组装完整提示词，调用 LLM。

In [ ]:
class SummaryBufferMemory:
    """混合记忆：滚动摘要 + 近期消息缓冲区。"""

    def __init__(
        self,
        model,
        system_prompt: str | None = None,
        max_buffer_tokens: int = 300,
    ):
        self.model = model
        self.system_prompt = system_prompt
        self.max_buffer_tokens = max_buffer_tokens

        # 两个记忆区域
        self.buffer: list[dict] = []      # 近期消息（逐字保存）
        self.summary: str = ""             # 压缩后的较旧历史

        # 记录用
        self.turn_count: int = 0
        self.eviction_log: list[dict] = []  # 记录每次淘汰事件


接下来加入淘汰和提示词构建逻辑。`_evict_if_needed` 每次从缓冲区弹出最旧的一条消息，然后调用 `update_summary` 把它们并入滚动摘要。`_build_messages` 把系统提示词、摘要和剩下的缓冲区拼成一条发给 API 的提示词。

In [ ]:
# 淘汰与提示词组装
def _evict_if_needed(self) -> None:
    """超出预算时，把最旧的消息从缓冲区移入摘要。"""
    evicted_batch = []

    while (
        len(self.buffer) > 2
        and count_message_tokens(self.buffer) > self.max_buffer_tokens
    ):
        # 淘汰最旧的消息（FIFO：先进先出）
        evicted_batch.append(self.buffer.pop(0))

    if evicted_batch:
        old_summary = self.summary
        self.summary = update_summary(self.summary, evicted_batch)

        self.eviction_log.append({
            "turn": self.turn_count,
            "evicted_count": len(evicted_batch),
            "summary_before": old_summary,
            "summary_after": self.summary,
        })


def _build_system_prompt(self) -> str:
    """把系统提示词和滚动摘要合成一段系统上下文。"""
    parts = []
    if self.system_prompt:
        parts.append(self.system_prompt)
    if self.summary:
        parts.append(f"Summary of earlier conversation:\n{self.summary}")
    return "\n\n".join(parts) or "You are a helpful assistant."


SummaryBufferMemory._evict_if_needed = _evict_if_needed
SummaryBufferMemory._build_system_prompt = _build_system_prompt

print("✓ eviction and prompt helpers attached")


`chat` 方法把所有环节串起来。它追加用户消息，按需触发淘汰，组装提示词并调用 LLM。模型的回应回到缓冲区，供后续上下文使用。

In [ ]:
# ── 对话 ──────────────────────────────────────────────────────
def _chat(self, user_input: str) -> str:
    """发送一条消息并获取回应。"""
    self.buffer.append({"role": "user", "content": user_input})
    self.turn_count += 1

    # 调用 LLM 前先检查是否需要淘汰
    self._evict_if_needed()

    # 摘要随淘汰更新，系统提示词每轮可能变化，所以每轮重建智能体
    agent = create_deep_agent(
        model=self.model,
        system_prompt=self._build_system_prompt(),
    )
    result = agent.invoke({"messages": list(self.buffer)})
    assistant_text = to_text(result["messages"][-1].content)

    self.buffer.append({"role": "assistant", "content": assistant_text})

    return assistant_text


SummaryBufferMemory.chat = _chat

print("✓ SummaryBufferMemory.chat attached")


最后加入几个用于查看和重置记忆的工具方法。`get_buffer` 和 `get_summary` 让你查看两个区域的内容。`get_buffer_token_count` 告诉你缓冲区离淘汰阈值还有多远。

In [ ]:
# ── 检查辅助方法 ──────────────────────────────────────────────
def _get_buffer(self) -> list[dict]:
    """返回当前缓冲区的副本。"""
    return copy.deepcopy(self.buffer)


def _get_summary(self) -> str:
    """返回当前的滚动摘要。"""
    return self.summary


def _get_buffer_token_count(self) -> int:
    """返回缓冲区当前的 token 数。"""
    return count_message_tokens(self.buffer)


def _clear(self) -> None:
    """重置两个记忆区域。"""
    self.buffer.clear()
    self.summary = ""
    self.turn_count = 0
    self.eviction_log.clear()


def _repr(self) -> str:
    return (
        f"SummaryBufferMemory("
        f"buffer={len(self.buffer)} msgs, "
        f"summary={'yes' if self.summary else 'empty'}, "
        f"turns={self.turn_count})"
    )


SummaryBufferMemory.get_buffer = _get_buffer
SummaryBufferMemory.get_summary = _get_summary
SummaryBufferMemory.get_buffer_token_count = _get_buffer_token_count
SummaryBufferMemory.clear = _clear
SummaryBufferMemory.__repr__ = _repr

print("✓ SummaryBufferMemory helper methods attached")


## 运行示例

我们来跑一段多轮对话，观察摘要缓冲的实际表现。缓冲阈值设小一点（300 token），让淘汰尽快发生。生产环境中你会把它设得更高。

创建记忆并跑一段对话。每轮之后打印智能体的回复。

In [ ]:
memory = SummaryBufferMemory(
    model=llm,
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
    max_buffer_tokens=300,
)

exchanges = [
    "Hi! My name is Alice and I'm a machine-learning engineer in Berlin.",
    "I'm building a chatbot for our customer support team. We get about 500 tickets a day.",
    "The main issue is that our current bot forgets context after 3 messages.",
    "We use Python and FastAPI on the backend. The bot runs on GPT-4o.",
    "Our customers mostly ask about billing, shipping, and returns.",
    "I also want the bot to remember user preferences across sessions.",
    "We store conversation logs in PostgreSQL right now.",
    "What memory technique would you recommend for our use case?",
    "By the way, what's my name and where do I work?",
]

for msg in exchanges:
    print(f"\n{'='*60}")
    print(f"Turn {memory.turn_count + 1}")
    print(f"{'='*60}")
    print(f"User:  {msg}")
    reply = memory.chat(msg)
    print(f"Agent: {reply}")
    print(f"\n  Buffer: {len(memory.buffer)} messages ({memory.get_buffer_token_count()} tokens)")
    print(f"  Summary: {'[empty]' if not memory.summary else memory.summary[:100] + '...'}")

### 查看记忆状态

9 轮之后，一些消息已经淘汰并纳入摘要。看看每个区域里现在有什么。

In [ ]:
print("RUNNING SUMMARY")
print("-" * 40)
print(memory.get_summary() or "(empty)")

print("\nBUFFER (recent messages)")
print("-" * 40)
for i, msg in enumerate(memory.get_buffer()):
    role_label = "USER" if msg["role"] == "user" else "ASST"
    preview = msg["content"][:90] + ("..." if len(msg["content"]) > 90 else "")
    print(f"  [{i}] {role_label}: {preview}")

print(f"\nBuffer token count: {memory.get_buffer_token_count()}")
print(f"Buffer threshold:   {memory.max_buffer_tokens}")

### 淘汰日志

消息每次移出缓冲区，系统都会记录这次事件。这份日志展示了摘要如何随时间演进。

In [ ]:
print(f"Total eviction events: {len(memory.eviction_log)}\n")

for i, event in enumerate(memory.eviction_log):
    print(f"Eviction {i + 1} (after turn {event['turn']})")
    print(f"  Messages evicted: {event['evicted_count']}")
    print(f"  Summary before: {event['summary_before'][:80] or '(empty)'}...")
    print(f"  Summary after:  {event['summary_after'][:80]}...")
    print()

### LLM 实际看到什么

我们看看组装好的提示词。这正是下一次调用时发给 API 的内容。注意摘要放在系统消息里，缓冲消息跟在后面。

In [ ]:
assembled = [{"role": "system", "content": memory._build_system_prompt()}] + memory.get_buffer()

print(f"Total messages in assembled prompt: {len(assembled)}")
print(f"Total tokens in assembled prompt:   {count_message_tokens(assembled)}\n")

for i, msg in enumerate(assembled):
    role = msg["role"].upper()
    content = msg["content"]
    if len(content) > 150:
        content = content[:150] + "..."
    print(f"[{i}] {role}:")
    print(f"    {content}")
    print()

## 对比：纯缓冲 vs. 摘要缓冲

和纯缓冲记忆相比，我们的混合方案在 token 用量上表现如何？用同一段对话跑两种方式，测一测。

In [ ]:
# 模拟纯缓冲记忆：假设每轮都发送全部消息来统计 token。
# 复用上面同一个 exchanges 列表。

all_messages = []  # 像纯缓冲一样不断累积
buffer_tokens_per_turn = []
summary_buffer_tokens_per_turn = []

# 重置摘要缓冲记忆，跑一次干净的对比
sbm = SummaryBufferMemory(
    model=llm,
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
    max_buffer_tokens=300,
)

for msg in exchanges:
    # 纯缓冲：累积全部消息
    all_messages.append({"role": "user", "content": msg})
    buffer_tokens_per_turn.append(count_message_tokens(all_messages))
    # 加一条占位的助手回复，让计数更接近真实
    all_messages.append({"role": "assistant", "content": "Acknowledged."})

    # 摘要缓冲：使用真正的类
    sbm.chat(msg)
    assembled_msgs = [{"role": "system", "content": sbm._build_system_prompt()}] + sbm.get_buffer()
    summary_buffer_tokens_per_turn.append(count_message_tokens(assembled_msgs))

print(f"{'Turn':<6} {'Buffer Memory':<18} {'Summary Buffer Memory'}")
print("-" * 48)
for i in range(len(exchanges)):
    print(f"{i+1:<6} {buffer_tokens_per_turn[i]:<18} {summary_buffer_tokens_per_turn[i]}")

print(f"\nFinal turn token usage:")
print(f"  Buffer memory:         {buffer_tokens_per_turn[-1]} tokens")
print(f"  Summary buffer memory: {summary_buffer_tokens_per_turn[-1]} tokens")
print(f"  Savings:               {buffer_tokens_per_turn[-1] - summary_buffer_tokens_per_turn[-1]} tokens")

## 权衡

### 摘要缓冲记忆的适用场景

- **中等长度到长对话**（20-100+ 轮），既需要历史背景又需要精确的近期细节。客服机器人和教练型智能体是常见例子。
- **对 token 预算敏感的应用**：摘要压缩了较早的历史，累计 token 用量比纯缓冲记忆增长慢得多。
- **近期上下文最重要时**：缓冲区完整保留最后几轮交互。模型可以引用近期轮次里的原话、数字和名字。

### 什么时候会失效

- **摘要质量随时间下降。** 每次增量摘要调用都有丢失细节的风险。几十次淘汰之后，摘要可能漏掉重要的事实。没有内置的办法验证摘要是否准确。
- **额外的 LLM 调用增加成本和延迟。** 每次淘汰都触发一次摘要调用。对节奏快的对话（要求亚秒级响应），这个开销可能太高。
- **比纯缓冲更难调试。** 智能体忘掉某件事时，你要追查信息是在摘要时丢的，还是根本没进过缓冲区。纯缓冲记忆里完整历史始终都在。
- **调参不容易。** 缓冲阈值、摘要提示词和摘要用的模型相互影响。找到合适的平衡需要实验。

## 延伸阅读

- [LangChain ConversationSummaryBufferMemory](https://python.langchain.com/docs/modules/memory/types/summary_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：这一模式的框架实现，支持可配置的 token 上限和基于 LLM 的摘要。
- [MemGPT / Letta: Tiered Memory Management (arXiv:2310.08560)](https://arxiv.org/abs/2310.08560)：Packer 等人提出多层记忆架构，让 LLM 管理自己的上下文窗口，在主上下文和外部存储之间搬运数据。
- [LlamaIndex ChatSummaryMemoryBuffer](https://docs.llamaindex.ai/en/stable/api_reference/memory/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：LlamaIndex 的实现，把摘要缓冲记忆接进检索增强生成（RAG）流程。
- [OpenAI Cookbook: Counting Tokens with tiktoken](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：聊天应用中统计和管理 token 的实用方法。

---

*← 上一篇：[03: 摘要记忆](../03_summary_memory/) · 下一篇：[05: token 缓冲记忆](../05_token_buffer_memory/) →*

## 🧪 动手试一试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：调整淘汰阈值
把 `SummaryBufferMemory` 的 token 阈值改成 500、1000 和 2000。每个设置下跑一段 15 轮的对话，记录 `_evict_if_needed()` 触发的频率，并记录每次最终的摘要长度。

### 挑战 2：追踪压缩比
每轮淘汰结束后，计算节省的 token 比例（淘汰的原始 token 数减去摘要 token 数）。把这个压缩比（compression ratio）随轮次画出来。找出摘要收益开始递减的转折点。

### 挑战 3：实体感知的摘要
调用 `update_summary()` 之前，先用 07 实体记忆的方法从淘汰的消息中抽取实体。把实体列表传给摘要提示词，让它保留命名实体。对比修改前后，模型在实体类问题上的查全率（recall）。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--04-summary-buffer-memory--summary-buffer-memory)
